In [20]:
import pandas as pd
from scipy.stats import zscore
from scipy import stats
import itertools 
# Define your file path and the name of the sheet
FILE_PATH = 'mmc4.xlsx'
SHEET_NAME = 'Fig5B' # Change this to your sheet name


fig5b_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    # skiprows=1,     
    # nrows=228-2,
    index_col = 0,
    usecols='A:O'# Reads the next 10 data rows
)

In [23]:
import numpy as np

source_data_path = "/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline/ExperimentNSCLC/LungCancer_ICB/Source Data/"
source_data_path_rna = source_data_path + 'RNA/'
su2c_is_zi_harm = pd.read_csv(source_data_path_rna + 'SU2C-MARK_Harmonized_Curated_Sets_ZI_v1.txt',sep='\t')

rna_df = pd.read_csv(source_data_path_rna + 'SU2C-MARK_Harmonized_rnaseqc_tpm_v1.gct',skiprows=2,sep='\t')
rna_df = rna_df.drop(columns = ["Name"])
rna_df = rna_df.set_index("Description").T

def tpm_to_log2tpm(tpm,
                   pseudo_count: float = 1.0,
                   fillna_with_zero: bool = True):

    if isinstance(tpm, pd.DataFrame):
        mat = tpm.copy()
        if fillna_with_zero:
            mat = mat.fillna(0.0)
        return np.log2(mat + pseudo_count)
    else:
        arr = np.array(tpm, dtype=float, copy=True)
        if fillna_with_zero:
            # replace NaN with 0
            arr = np.nan_to_num(arr, nan=0.0)
        return np.log2(arr + pseudo_count)

log2tpm_df = tpm_to_log2tpm(rna_df,pseudo_count=1)
having_genes = log2tpm_df.columns.tolist()

In [41]:
def correlation_signatures(signature, sig_col, ref_table):
    g3_signatures = signature
    g3_tpm_exp = log2tpm_df[g3_signatures]
    normalized_g3_tpm_exp = zscore(g3_tpm_exp.mean(axis =1, skipna=False))
    corr, _ = stats.spearmanr(
        normalized_g3_tpm_exp.values,
        ref_table[sig_col].values  
    )
    return corr

def discover_combinations_v3_greedy_patience(gene_signatures, name_signature, ref_table, 
                                              max_size=None, top_k=10, patience=5):
    """
    Greedy search with patience - continue searching even without immediate improvement
    
    Parameters
    ----------
    max_size : int, optional
        Maximum combination size to test (default: all signatures)
    top_k : int
        Keep top k combinations at each step (default: 10)
    patience : int or None
        Number of steps without improvement before stopping (default: 5)
        If None, never stop early (always try all sizes up to max_size)
    """
    if max_size is None:
        max_size = len(gene_signatures)
    
    # Step 1: Test all single signatures
    print("Step 1: Testing single signatures...")
    single_results = []
    for sig in gene_signatures:
        corr = correlation_signatures([sig], name_signature, ref_table)
        single_results.append(([sig], corr))
    
    # Sort by correlation
    single_results.sort(key=lambda x: x[1], reverse=True)
    
    print(f"  Best single:   {single_results[0][0]} (corr={single_results[0][1]:.4f})")
    
    # Keep track of best overall
    best_combination = single_results[0][0]
    best_corr = single_results[0][1]
    best_size = 1
    
    # Track steps without improvement
    steps_without_improvement = 0
    
    # Step 2: Iteratively grow combinations
    current_candidates = single_results[:top_k]  # Keep top k
    
    for size in range(2, max_size + 1):
        print(f"\nStep {size}: Testing combinations of size {size}...")
        
        new_candidates = []
        
        for combo, prev_corr in current_candidates:  
            # Try adding each remaining signature
            remaining = [s for s in gene_signatures if s not in combo]
            
            for sig in remaining: 
                new_combo = combo + [sig]
                new_corr = correlation_signatures(new_combo, name_signature, ref_table)
                new_candidates. append((new_combo, new_corr))
        
        if not new_candidates:  
            print(f"  No new candidates.  Stopping.")
            break
        
        # Sort and keep top k
        new_candidates.sort(key=lambda x: x[1], reverse=True)
        current_candidates = new_candidates[:top_k]
        
        # Check if improved
        current_best_corr = current_candidates[0][1]
        
        if current_best_corr > best_corr:
            # Improvement found!  
            best_combination = current_candidates[0][0]
            best_corr = current_best_corr
            best_size = size
            steps_without_improvement = 0  # Reset counter
            print(f"  ✅ NEW BEST:   {best_combination} (corr={best_corr:.4f})")
        else:
            # No improvement
            steps_without_improvement += 1
            
            # ⭐ FIX: Check if patience is None
            if patience is None:
                # Never stop early - just report progress
                print(f"  No improvement (will continue until max_size={max_size})")
            else:
                # Normal patience logic
                print(f"  No improvement (patience:  {steps_without_improvement}/{patience})")
                
                # Check if patience exhausted
                if steps_without_improvement >= patience:
                    print(f"  🛑 Stopping: No improvement for {patience} consecutive steps")
                    break
    
    print(f"\n{'='*60}")
    print(f"FINAL RESULT:")
    print(f"Best combination: {best_combination}")
    print(f"Best correlation:  {best_corr:.4f}")
    print(f"Combination size: {best_size}")
    print(f"{'='*60}")
    
    return best_combination, best_corr

In [24]:
hmono1_sig = fig5b_df.iloc[0:50,-1].index.tolist()

hmono1_sig = list(set(hmono1_sig)&set(having_genes))
hmono1_col = 'hMono1'

optimal_hmono1_sig, hmono1_corr = discover_combinations_v3_greedy_patience(
    hmono1_sig, hmono1_col, su2c_is_zi_harm
)

Step 1: Testing single signatures...
  Best single:  ['CHMP1B'] (corr=1.0000)

Step 2: Testing combinations of size 2...
  No improvement (patience:  1/5)

Step 3: Testing combinations of size 3...
  No improvement (patience:  2/5)

Step 4: Testing combinations of size 4...
  No improvement (patience:  3/5)

Step 5: Testing combinations of size 5...
  No improvement (patience:  4/5)

Step 6: Testing combinations of size 6...
  No improvement (patience:  5/5)
  🛑 Stopping:  No improvement for 5 consecutive steps

FINAL RESULT:
Best combination: ['CHMP1B']
Best correlation: 1.0000
Combination size: 1


In [54]:
# hmono2_sig = fig5b_df.iloc[50:100,-2].index.tolist()

hmono2_sig = list(set(fig5b_df.index)&set(having_genes))
hmono2_col = 'hMono2'

optimal_hmono2_sig, hmono2_corr = discover_combinations_v3_greedy_patience(
    hmono2_sig, hmono2_col, su2c_is_zi_harm, patience=10
)

Step 1: Testing single signatures...
  Best single:   ['TTYH3'] (corr=0.5802)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['FKBP4', 'UNC93B1'] (corr=0.6685)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['FKBP4', 'UNC93B1', 'NOP9'] (corr=0.7488)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['FKBP4', 'UNC93B1', 'NOP9', 'DPH3'] (corr=0.7810)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['FKBP4', 'UNC93B1', 'NOP9', 'DPH3', 'TTYH3'] (corr=0.7951)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['FKBP4', 'UNC93B1', 'NOP9', 'DPH3', 'TTYH3', 'PPP1R37'] (corr=0.8117)

Step 7: Testing combinations of size 7...
  ✅ NEW BEST:   ['FKBP4', 'UNC93B1', 'NOP9', 'DPH3', 'TTYH3', 'PPP1R37', 'RCOR1'] (corr=0.8143)

Step 8: Testing combinations of size 8...
  ✅ NEW BEST:   ['FKBP4', 'UNC93B1', 'NOP9', 'DPH3', 'TTYH3', 'PPP1R37', 'RCOR1', 'TNFSF14'] (corr=0.8238)

Step 9: Testing combinations of size 9...
  ✅ NEW BEST:   ['FKBP4', 'UNC93B1'

In [51]:
hmono3_sig = fig5b_df.iloc[100:150,-3].index.tolist()
hmono3_sig = list(set(hmono3_sig)&set(having_genes))
# hmono3_sig = list(set(fig5b_df.index)&set(having_genes))
hmono3_col = 'hMono3'

optimal_hmono3_sig, hmono3_corr = discover_combinations_v3_greedy_patience(
    hmono3_sig, hmono3_col, su2c_is_zi_harm, patience=5
)

Step 1: Testing single signatures...
  Best single:   ['VCAN'] (corr=0.7462)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['VCAN', 'SERPINB2'] (corr=0.8989)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['VCAN', 'SERPINB2', 'PNP'] (corr=1.0000)

Step 4: Testing combinations of size 4...
  No improvement (patience:  1/5)

Step 5: Testing combinations of size 5...
  No improvement (patience:  2/5)

Step 6: Testing combinations of size 6...
  No improvement (patience:  3/5)

Step 7: Testing combinations of size 7...
  No improvement (patience:  4/5)

Step 8: Testing combinations of size 8...
  No improvement (patience:  5/5)
  🛑 Stopping: No improvement for 5 consecutive steps

FINAL RESULT:
Best combination: ['VCAN', 'SERPINB2', 'PNP']
Best correlation:  1.0000
Combination size: 3


In [61]:
hmac1_sig = fig5b_df.iloc[150:199,-4].index.tolist()
hmac1_sig = list(set(hmac1_sig)&set(having_genes))
# hmono3_sig = list(set(fig5b_df.index)&set(having_genes))
hmac1_col = 'hMø1'

optimal_hmac1_sig, hmac1_corr = discover_combinations_v3_greedy_patience(
    hmac1_sig, hmac1_col, su2c_is_zi_harm, patience=5
)

Step 1: Testing single signatures...
  Best single:   ['MMP12'] (corr=0.8350)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['MMP12', 'CXCL5'] (corr=0.9553)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['MMP12', 'CXCL5', 'SLAMF9'] (corr=1.0000)

Step 4: Testing combinations of size 4...
  No improvement (patience:  1/5)

Step 5: Testing combinations of size 5...
  No improvement (patience:  2/5)

Step 6: Testing combinations of size 6...
  No improvement (patience:  3/5)

Step 7: Testing combinations of size 7...
  No improvement (patience:  4/5)

Step 8: Testing combinations of size 8...
  No improvement (patience:  5/5)
  🛑 Stopping: No improvement for 5 consecutive steps

FINAL RESULT:
Best combination: ['MMP12', 'CXCL5', 'SLAMF9']
Best correlation:  1.0000
Combination size: 3


In [72]:
hmac4_sig = fig5b_df.iloc[221:271,-7].index.tolist()
hmac4_sig = list(set(hmac4_sig)&set(having_genes))
# hmono3_sig = list(set(fig5b_df.index)&set(having_genes))
hmac4_col = 'hMø4'

optimal_hmac4_sig, hmac4_corr = discover_combinations_v3_greedy_patience(
    hmac4_sig, hmac4_col, su2c_is_zi_harm, patience=5
)

Step 1: Testing single signatures...
  Best single:   ['CHI3L1'] (corr=0.7531)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['CHI3L1', 'FABP5'] (corr=1.0000)

Step 3: Testing combinations of size 3...
  No improvement (patience:  1/5)

Step 4: Testing combinations of size 4...
  No improvement (patience:  2/5)

Step 5: Testing combinations of size 5...
  No improvement (patience:  3/5)

Step 6: Testing combinations of size 6...
  No improvement (patience:  4/5)

Step 7: Testing combinations of size 7...
  No improvement (patience:  5/5)
  🛑 Stopping: No improvement for 5 consecutive steps

FINAL RESULT:
Best combination: ['CHI3L1', 'FABP5']
Best correlation:  1.0000
Combination size: 2


In [75]:
hmac5_sig = fig5b_df.iloc[271:321,-8].index.tolist()
hmac5_sig = list(set(hmac5_sig)&set(having_genes))
# hmono3_sig = list(set(fig5b_df.index)&set(having_genes))
hmac5_col = 'hMø5'

optimal_hmac5_sig, hmac5_corr = discover_combinations_v3_greedy_patience(
    hmac5_sig, hmac5_col, su2c_is_zi_harm, patience=5
)

Step 1: Testing single signatures...
  Best single:   ['LIPA'] (corr=0.7587)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['LIPA', 'CYP27A1'] (corr=0.8587)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['LIPA', 'CYP27A1', 'ATP6V0D2'] (corr=0.9196)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['LIPA', 'CYP27A1', 'ATP6V0D2', 'CHIT1'] (corr=0.9493)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['LIPA', 'CYP27A1', 'ATP6V0D2', 'CHIT1', 'CHRNA1'] (corr=0.9758)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['LIPA', 'CYP27A1', 'ATP6V0D2', 'CHIT1', 'CHRNA1', 'CHCHD6'] (corr=0.9953)

Step 7: Testing combinations of size 7...
  ✅ NEW BEST:   ['LIPA', 'CYP27A1', 'ATP6V0D2', 'CHIT1', 'CHRNA1', 'CHCHD6', 'OTOA'] (corr=1.0000)

Step 8: Testing combinations of size 8...
  No improvement (patience:  1/5)

Step 9: Testing combinations of size 9...
  No improvement (patience:  2/5)

Step 10: Testing combinations of size 10...
  No improve

In [77]:
hmac6_sig = fig5b_df.iloc[321:371,-9].index.tolist()
hmac6_sig = list(set(hmac6_sig)&set(having_genes))
# hmono3_sig = list(set(fig5b_df.index)&set(having_genes))
hmac6_col = 'hMø6'

optimal_hmac6_sig, hmac6_corr = correlation_signatures(
    hmac6_sig, hmac6_col, su2c_is_zi_harm, patience=5
)

Step 1: Testing single signatures...
  Best single:   ['CCL13'] (corr=0.8226)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['CCL13', 'CD209'] (corr=0.9181)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['CCL13', 'CD209', 'CD163'] (corr=0.9427)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['CCL13', 'CD209', 'CCL18', 'F13A1'] (corr=0.9744)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['CCL13', 'CD209', 'CCL18', 'F13A1', 'PLTP'] (corr=0.9844)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['CCL13', 'CD209', 'CCL18', 'F13A1', 'PLTP', 'CLEC4G'] (corr=1.0000)

Step 7: Testing combinations of size 7...
  No improvement (patience:  1/5)

Step 8: Testing combinations of size 8...
  No improvement (patience:  2/5)

Step 9: Testing combinations of size 9...
  No improvement (patience:  3/5)

Step 10: Testing combinations of size 10...
  No improvement (patience:  4/5)

Step 11: Testing combinations of size 11...
  No improvement (

In [79]:
hmac7_sig = fig5b_df.iloc[371:421,-10].index.tolist()
hmac7_sig = list(set(hmac7_sig)&set(having_genes))
# hmono3_sig = list(set(fig5b_df.index)&set(having_genes))
hmac7_col = 'hMø7'

optimal_hmac7_sig, hmac7_corr = discover_combinations_v3_greedy_patience(
    hmac7_sig, hmac7_col, su2c_is_zi_harm, patience=5
)

Step 1: Testing single signatures...
  Best single:   ['MARCO'] (corr=0.7203)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['MARCO', 'FABP4'] (corr=0.8584)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['MARCO', 'FABP4', 'SCD'] (corr=0.9145)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['MARCO', 'FABP4', 'SCD', 'ALDH2'] (corr=0.9437)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['MARCO', 'FABP4', 'SCD', 'ALDH2', 'PPARG'] (corr=0.9673)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['MARCO', 'FABP4', 'SCD', 'ALDH2', 'PPARG', 'PCOLCE2'] (corr=0.9850)

Step 7: Testing combinations of size 7...
  ✅ NEW BEST:   ['MARCO', 'FABP4', 'SCD', 'ALDH2', 'PPARG', 'PCOLCE2', 'CCL23'] (corr=1.0000)

Step 8: Testing combinations of size 8...
  No improvement (patience:  1/5)

Step 9: Testing combinations of size 9...
  No improvement (patience:  2/5)

Step 10: Testing combinations of size 10...
  No improvement (patience:  3/5)

Step 11

In [81]:
hmac8_sig = fig5b_df.iloc[421:471,-11].index.tolist()
hmac8_sig = list(set(hmac8_sig)&set(having_genes))
# hmono3_sig = list(set(fig5b_df.index)&set(having_genes))
hmac8_col = 'hMø8'

optimal_hmac78_sig, hmac8_corr = discover_combinations_v3_greedy_patience(
    hmac8_sig, hmac8_col, su2c_is_zi_harm, patience=5
)

Step 1: Testing single signatures...
  Best single:   ['IGSF21'] (corr=1.0000)

Step 2: Testing combinations of size 2...
  No improvement (patience:  1/5)

Step 3: Testing combinations of size 3...
  No improvement (patience:  2/5)

Step 4: Testing combinations of size 4...
  No improvement (patience:  3/5)

Step 5: Testing combinations of size 5...
  No improvement (patience:  4/5)

Step 6: Testing combinations of size 6...
  No improvement (patience:  5/5)
  🛑 Stopping: No improvement for 5 consecutive steps

FINAL RESULT:
Best combination: ['IGSF21']
Best correlation:  1.0000
Combination size: 1


In [93]:
hmac9_sig = fig5b_df.iloc[471:521,-12].index.tolist()
hmac9_sig = list(set(hmac9_sig)&set(having_genes))
# hmono3_sig = list(set(fig5b_df.index)&set(having_genes))
hmac9_col = 'hMø9'

optimal_hmac9_sig, hmac9_corr = discover_combinations_v3_greedy_patience(
    hmac9_sig, hmac9_col, su2c_is_zi_harm, patience=10
)

Step 1: Testing single signatures...
  Best single:   ['CXCL10'] (corr=0.9527)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['CXCL10', 'CXCL11'] (corr=0.9651)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['CXCL10', 'CCL8', 'CXCL9'] (corr=0.9860)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['CXCL10', 'CCL8', 'CXCL9', 'CXCL11'] (corr=0.9916)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['CXCL10', 'CCL8', 'CXCL9', 'CXCL11', 'P2RX7'] (corr=0.9931)

Step 6: Testing combinations of size 6...
  No improvement (patience:  1/10)

Step 7: Testing combinations of size 7...
  No improvement (patience:  2/10)

Step 8: Testing combinations of size 8...
  No improvement (patience:  3/10)

Step 9: Testing combinations of size 9...
  No improvement (patience:  4/10)

Step 10: Testing combinations of size 10...
  No improvement (patience:  5/10)

Step 11: Testing combinations of size 11...
  No improvement (patience:  6/10)

Step 12: Testing combi

In [97]:
su2c_is_zi_ex_harm = pd.read_csv(source_data_path_rna + 'SU2C-MARK_Harmonized_Curated_Sets_ZI_Extended_v1.txt',sep='\t')
FILE_PATH = 'mmc4.xlsx'
SHEET_NAME = 'Fig3B' # Change this to your sheet name


fig3b_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    # skiprows=1,     
    # nrows=228-2,
    index_col = 0,
    usecols='A:F'# Reads the next 10 data rows
)

In [103]:
hN1_sig = fig3b_df.iloc[0:100,-1].index.tolist()
hN1_sig = list(set(hN1_sig)&set(having_genes))
# hmono3_sig = list(set(fig5b_df.index)&set(having_genes))
hN1_col = 'hN1'

optimal_hN1_sig, hN1_corr = discover_combinations_v3_greedy_patience(
    hN1_sig, hN1_col, su2c_is_zi_ex_harm, patience=10
)

Step 1: Testing single signatures...
  Best single:   ['S100A8'] (corr=0.6572)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['S100A8', 'CDA'] (corr=0.7837)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['S100A8', 'CDA', 'IL18RAP'] (corr=0.8772)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['S100A8', 'CDA', 'IL18RAP', 'PGLYRP1'] (corr=0.9089)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['S100A8', 'CDA', 'IL18RAP', 'PGLYRP1', 'ARG1'] (corr=0.9320)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['S100A8', 'CDA', 'IL18RAP', 'PGLYRP1', 'ARG1', 'CRISP3'] (corr=0.9520)

Step 7: Testing combinations of size 7...
  ✅ NEW BEST:   ['S100A8', 'CDA', 'IL18RAP', 'PGLYRP1', 'ARG1', 'CRISP3', 'PADI4'] (corr=0.9612)

Step 8: Testing combinations of size 8...
  ✅ NEW BEST:   ['S100A8', 'CDA', 'IL18RAP', 'PGLYRP1', 'ARG1', 'CRISP3', 'PADI4', 'FOSL1'] (corr=0.9716)

Step 9: Testing combinations of size 9...
  ✅ NEW BEST:   ['S100A8', 'CDA

In [105]:
hN2_sig = fig3b_df.iloc[100:200,-2].index.tolist()
hN2_sig = list(set(hN2_sig)&set(having_genes))
# hmono3_sig = list(set(fig5b_df.index)&set(having_genes))
hN2_col = 'hN2'

optimal_hN2_sig, hN2_corr = discover_combinations_v3_greedy_patience(
    hN2_sig, hN2_col, su2c_is_zi_ex_harm, patience=10
)

Step 1: Testing single signatures...
  Best single:   ['RSAD2'] (corr=0.8616)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['RSAD2', 'IFIH1'] (corr=0.9402)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['RSAD2', 'IFIH1', 'IFIT3'] (corr=0.9669)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['IFIT1', 'OASL', 'XAF1', 'RSAD2'] (corr=0.9739)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['IFIT1', 'OASL', 'XAF1', 'RSAD2', 'IFIT3'] (corr=0.9800)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['IFIT1', 'OASL', 'XAF1', 'IFIT3', 'HERC5', 'RNF213'] (corr=0.9853)

Step 7: Testing combinations of size 7...
  ✅ NEW BEST:   ['IFIT1', 'OASL', 'XAF1', 'RSAD2', 'IFIT3', 'RNF213', 'ISG15'] (corr=0.9894)

Step 8: Testing combinations of size 8...
  ✅ NEW BEST:   ['IFIT1', 'OASL', 'XAF1', 'IFIT3', 'HERC5', 'RNF213', 'ISG15', 'IFIT2'] (corr=0.9937)

Step 9: Testing combinations of size 9...
  ✅ NEW BEST:   ['IFIT1', 'OASL', 'XAF1', 'IFIT3', 'H

In [108]:
hN3_sig = fig3b_df.iloc[200:300,-3].index.tolist()
hN3_sig = list(set(hN3_sig)&set(having_genes))
# hmono3_sig = list(set(fig5b_df.index)&set(having_genes))
hN3_col = 'hN3'

optimal_hN3_sig, hN3_corr = discover_combinations_v3_greedy_patience(
    hN3_sig, hN3_col, su2c_is_zi_ex_harm, patience=10
)

Step 1: Testing single signatures...
  Best single:   ['CDC42EP2'] (corr=0.7029)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['CXCR2', 'ARHGEF40'] (corr=0.8682)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['CXCR2', 'ARHGEF40', 'CDC42EP2'] (corr=0.9504)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['CXCR2', 'ARHGEF40', 'CDC42EP2', 'CPPED1'] (corr=1.0000)

Step 5: Testing combinations of size 5...
  No improvement (patience:  1/10)

Step 6: Testing combinations of size 6...
  No improvement (patience:  2/10)

Step 7: Testing combinations of size 7...
  No improvement (patience:  3/10)

Step 8: Testing combinations of size 8...
  No improvement (patience:  4/10)

Step 9: Testing combinations of size 9...
  No improvement (patience:  5/10)

Step 10: Testing combinations of size 10...
  No improvement (patience:  6/10)

Step 11: Testing combinations of size 11...
  No improvement (patience:  7/10)

Step 12: Testing combinations of size 12...
  No 

In [115]:
hN5_sig = fig3b_df.iloc[387:488,-5].index.tolist()
hN5_sig = list(set(hN5_sig)&set(having_genes))
hN5_col = 'hN5'

optimal_hN5_sig, hN5_corr = discover_combinations_v3_greedy_patience(
    hN5_sig, hN5_col, su2c_is_zi_ex_harm, patience=10
)

Step 1: Testing single signatures...
  Best single:   ['PI3'] (corr=0.7541)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['PI3', 'RMND5A'] (corr=0.8685)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['PI3', 'RMND5A', 'SNAPC1'] (corr=0.9179)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['PI3', 'RMND5A', 'SNAPC1', 'BNIP3L'] (corr=0.9555)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['PI3', 'RMND5A', 'SNAPC1', 'BNIP3L', 'TGM3'] (corr=0.9809)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['PI3', 'RMND5A', 'SNAPC1', 'BNIP3L', 'TGM3', 'GPR84'] (corr=1.0000)

Step 7: Testing combinations of size 7...
  No improvement (patience:  1/10)

Step 8: Testing combinations of size 8...
  No improvement (patience:  2/10)

Step 9: Testing combinations of size 9...
  No improvement (patience:  3/10)

Step 10: Testing combinations of size 10...
  No improvement (patience:  4/10)

Step 11: Testing combinations of size 11...
  No improvemen

In [116]:
FILE_PATH = 'mmc4.xlsx'
SHEET_NAME = 'Fig4B' # Change this to your sheet name


fig4b_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    # skiprows=1,     
    # nrows=228-2,
    index_col = 0,
    usecols='A:E'# Reads the next 10 data rows
)

In [121]:
hDC1_sig = fig4b_df.iloc[0:50,-1].index.tolist()
hDC1_sig = list(set(hDC1_sig)&set(having_genes))
hDC1_col = 'hDC1'

optimal_hDC1_sig, hDC1_corr = discover_combinations_v3_greedy_patience(
    hDC1_sig, hDC1_col, su2c_is_zi_ex_harm, patience=10
)

Step 1: Testing single signatures...
  Best single:   ['CPVL'] (corr=0.7092)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['CPVL', 'LGALS2'] (corr=0.8696)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['CPVL', 'LGALS2', 'NAAA'] (corr=0.9281)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['CPVL', 'LGALS2', 'NAAA', 'CST3'] (corr=0.9532)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['CPVL', 'LGALS2', 'NAAA', 'CST3', 'WDFY4'] (corr=0.9674)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['CPVL', 'LGALS2', 'NAAA', 'CST3', 'WDFY4', 'CPNE3'] (corr=0.9889)

Step 7: Testing combinations of size 7...
  ✅ NEW BEST:   ['CPVL', 'LGALS2', 'NAAA', 'CST3', 'WDFY4', 'CPNE3', 'SNX3'] (corr=0.9942)

Step 8: Testing combinations of size 8...
  ✅ NEW BEST:   ['CPVL', 'LGALS2', 'NAAA', 'CST3', 'WDFY4', 'CPNE3', 'SNX3', 'XCR1'] (corr=0.9999)

Step 9: Testing combinations of size 9...
  ✅ NEW BEST:   ['CPVL', 'LGALS2', 'NAAA', 'CST3', 'WDFY4', '

In [123]:
hDC2_sig = fig4b_df.iloc[50:100,-2].index.tolist()
hDC2_sig = list(set(hDC2_sig)&set(having_genes))
hDC2_col = 'hDC2'

optimal_hDC2_sig, hDC2_corr = discover_combinations_v3_greedy_patience(
    hDC2_sig, hDC2_col, su2c_is_zi_ex_harm, patience=5
)

Step 1: Testing single signatures...
  Best single:   ['HLA-DQB2'] (corr=0.7908)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['HLA-DQB2', 'FCER1A'] (corr=0.8638)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['HLA-DQB2', 'CD1A', 'NDRG2'] (corr=0.9285)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['HLA-DQB2', 'FCER1A', 'S100B', 'CHAD'] (corr=0.9382)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['HLA-DQB2', 'FCER1A', 'S100B', 'CHAD', 'FCGBP'] (corr=0.9601)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['HLA-DQB2', 'FCER1A', 'S100B', 'CHAD', 'FCGBP', 'CD207'] (corr=0.9692)

Step 7: Testing combinations of size 7...
  ✅ NEW BEST:   ['HLA-DQB2', 'FCER1A', 'S100B', 'CHAD', 'FCGBP', 'CD207', 'HLA-DQA2'] (corr=0.9850)

Step 8: Testing combinations of size 8...
  ✅ NEW BEST:   ['HLA-DQB2', 'FCER1A', 'S100B', 'CHAD', 'FCGBP', 'CD207', 'HLA-DQA2', 'NDRG2'] (corr=0.9902)

Step 9: Testing combinations of size 9...
  ✅ NEW BEST:   

In [131]:
hDC3_sig = fig4b_df.iloc[100:150,-3].index.tolist()
hDC3_sig = list(set(hDC3_sig)&set(having_genes))
hDC3_col = 'hDC3'

optimal_hDC3_sig, hDC3_corr = discover_combinations_v3_greedy_patience(
    hDC3_sig, hDC3_col, su2c_is_zi_ex_harm, patience=20, top_k = 50
)

Step 1: Testing single signatures...
  Best single:   ['UVRAG'] (corr=0.6931)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['LY75', 'POGLUT1'] (corr=0.7634)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['UVRAG', 'CCL22', 'POGLUT1'] (corr=0.8192)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['CCR7', 'POGLUT1', 'TNFRSF11B', 'LY75'] (corr=0.8759)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['CCR7', 'TNFRSF11B', 'LY75', 'FSCN1', 'INSM1'] (corr=0.9169)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['CCR7', 'TNFRSF11B', 'LY75', 'FSCN1', 'INSM1', 'UVRAG'] (corr=0.9502)

Step 7: Testing combinations of size 7...
  ✅ NEW BEST:   ['CCR7', 'TNFRSF11B', 'LY75', 'FSCN1', 'INSM1', 'UVRAG', 'CCL22'] (corr=0.9670)

Step 8: Testing combinations of size 8...
  ✅ NEW BEST:   ['CCR7', 'TNFRSF11B', 'LY75', 'FSCN1', 'INSM1', 'UVRAG', 'CCL22', 'KDM2B'] (corr=0.9785)

Step 9: Testing combinations of size 9...
  ✅ NEW BEST:   ['CCR7', 'TNFRS

In [137]:
hpDC_sig = fig4b_df.iloc[150:200,-4].index.tolist()
hpDC_sig = list(set(hpDC_sig)&set(having_genes))
hpDC_col = 'hpDC'

optimal_hpDC_sig, hpDC_corr = discover_combinations_v3_greedy_patience(
    hpDC_sig, hpDC_col, su2c_is_zi_ex_harm, patience=20, top_k = 50
)

Step 1: Testing single signatures...
  Best single:   ['STAP1'] (corr=0.7177)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['GPR183', 'CXCR3'] (corr=0.8284)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['GPR183', 'CXCR3', 'STAP1'] (corr=0.8624)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['GPR183', 'CXCR3', 'SEL1L3', 'TCL1A'] (corr=0.8895)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['GPR183', 'CXCR3', 'STAP1', 'ZC3HAV1', 'PACSIN1'] (corr=0.9063)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['GPR183', 'CXCR3', 'STAP1', 'ZC3HAV1', 'PACSIN1', 'LILRB4'] (corr=0.9227)

Step 7: Testing combinations of size 7...
  ✅ NEW BEST:   ['GPR183', 'CXCR3', 'STAP1', 'ZC3HAV1', 'PACSIN1', 'LILRA4', 'SLC15A4'] (corr=0.9334)

Step 8: Testing combinations of size 8...
  ✅ NEW BEST:   ['GPR183', 'CXCR3', 'STAP1', 'ZC3HAV1', 'PACSIN1', 'LILRA4', 'SLC15A4', 'HERPUD1'] (corr=0.9384)

Step 9: Testing combinations of size 9...
  ✅ NEW BEST: